# 🎙️ GPT-SoVITS Colab FastBoot — `v0.3.1`
> **Version**: `v0.3.1` | **Model Profile**: `v2` / `v2Pro` / `v2ProPlus` + `v5turbo` / `v5dev` | **Target**: Colab Tesla T4 (16 GB VRAM)

针对 Colab 算力额度与启动耗时深度重构：
- **极速开局提速 3 大头**：
  1. `condacolab.install()` 轻量 Miniconda 替代 1GB 完整 Anaconda3（省 ~90 秒）。
  2. 通过 `PIP_CONSTRAINT` 单次直装 `torch==2.11.0+cu126` + `torchaudio==2.11.0+cu126` + `fastapi==0.115.12`，彻底杜绝 PyTorch 重复装两遍（省 ~45 秒）。
  3. 按需精简底模：默认拉取 **完整 `v2` 系列（`v2` / `v2Pro` / `v2ProPlus`）+ `v5`（`v5turbo` / `v5dev`）**，保留全部测试切换能力，仅剔除老旧 `v1` 与已弃用 `v3`/`v4`。
- **云盘缓存 & 显式计时**：单卡片原地进度条（不霸屏）+ 分阶段耗时表 + 一键备份/试听输出音频。

### 📊 GPT-SoVITS 各版本特点速查
| 版本 | 架构特点 | T4 (16GB) 表现 | 定位建议 |
| :--- | :--- | :--- | :--- |
| **`v1`** | 初代基准版 | 轻量 | 已过时，不建议新用 |
| **`v2`** | 5k 小时多语种（中英日韩粤），生态最老牌 | 推理 3-5GB / 训练 6-8GB | 架构偏旧，已被 `v2Pro` 替代 |
| **`v2Pro / v2ProPlus`** | 引入 `sv` 说话人编码器，音色相似度超越 v4 | 推理极快 / 微调稳跑 (8-10GB) | **T4 主力首选**（兼顾相似度、速度与微调友好度） |
| **`v3` / `v4`** | CFM 流匹配过渡代（含高显存 Vocoder） | 全量训练 ~14GB 易 OOM | **官方已标弃用**，已被 `v2Pro` 与 `v5` 取代 |
| **`v5dev`** | 新代 Vocoder 压制高频金属音/伪影（32 步采样） | 推理流畅（微调需调小 batch） | **追求最高音质细节**的新版推理首选 |
| **`v5turbo`** | 基于 v5 的 4 步 DMD 极速蒸馏，支持 `cuda_graph` | 推理极快 | **追求生成速度 + 新代音质**的首选 |

## Step 1. 硬件体检 & Google Drive 云盘缓存配置

In [ ]:
#@title 🔍 1. 检查 GPU/磁盘 并配置 Google Drive 云盘缓存 { display-mode: "form" }
ENABLE_GDRIVE_CACHE = True #@param {type:"boolean"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/GPT-SoVITS-Workspace" #@param {type:"string"}
CACHE_MODELS_TO_DRIVE = False #@param {type:"boolean"}
#@markdown > 注：Colab 从 HF 拉取 v2Pro 系列底模仅需 ~10-15 秒，通常比从 Drive 拷贝更快；仅在 HF 限速或需离线备份时勾选 `CACHE_MODELS_TO_DRIVE`。

import os, json, time, shutil, subprocess
from IPython.display import display, HTML

t0 = time.perf_counter()

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
except Exception as e:
    gpu_info = f"⚠️ 未检测到 NVIDIA GPU ({e}) — 请在 [代码执行程序 -> 更改运行时类型] 中选择 T4 GPU"

total_c, used_c, free_c = shutil.disk_usage("/content")
local_disk_str = f"{free_c/1e9:.1f} GB 可用 / {total_c/1e9:.1f} GB 总计"

drive_status = "未启用云盘缓存"
if ENABLE_GDRIVE_CACHE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    for sub in ["outputs", "ref_audios", "weights_backup", "hf_cache"]:
        os.makedirs(os.path.join(DRIVE_WORKSPACE, sub), exist_ok=True)
    if CACHE_MODELS_TO_DRIVE:
        os.makedirs(os.path.join(DRIVE_WORKSPACE, "models_v2pro_cache"), exist_ok=True)
    total_d, used_d, free_d = shutil.disk_usage("/content/drive/MyDrive")
    drive_status = f"✅ 已挂载 ({free_d/1e9:.1f} GB 可用 / {total_d/1e9:.1f} GB 总计) -> <code>{DRIVE_WORKSPACE}</code>"

cfg = {
    "enable_gdrive_cache": ENABLE_GDRIVE_CACHE,
    "drive_workspace": DRIVE_WORKSPACE,
    "cache_models": CACHE_MODELS_TO_DRIVE,
    "hf_cache": os.path.join(DRIVE_WORKSPACE, "hf_cache") if ENABLE_GDRIVE_CACHE else ""
}
with open("/content/.gsv_colab_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

elapsed = time.perf_counter() - t0
display(HTML(f"""
<div style='border:1px solid #38bdf8;border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
  <h4 style='margin:0 0 8px 0;color:#38bdf8;'>🖥️ 环境体检完成 (耗时 {elapsed:.2f}s)</h4>
  <div><b>GPU:</b> {gpu_info}</div>
  <div><b>Colab 本地空间:</b> {local_disk_str}</div>
  <div><b>云盘缓存:</b> {drive_status}</div>
</div>
"""))

## Step 2. 极速环境初始化（轻量 Miniconda + 单次依赖安装 + v2 系列底模）
> **重要提示**：首次运行 **2A** 安装 `condacolab` 时会自动重启一次内核——Colab 顶部若弹出 **“Your session crashed / 会话因未知原因崩溃”** 属于 `condacolab` **正常重启接管内核的现象**，点关闭提示后直接运行 **2B** 即可。

In [ ]:
#@title ⚙️ 2A. 初始化轻量 Condacolab（约 25-40 秒，弹窗提示重启属正常） { display-mode: "form" }
import os, time, shutil
t0 = time.perf_counter()
if os.path.exists("/usr/local/conda-meta") and shutil.which("conda"):
    print(f"✅ Conda 已就绪，跳过安装 (耗时 {time.perf_counter()-t0:.2f}s)")
else:
    print("⏳ 正在安装轻量版 condacolab (Miniconda，预计 25-40 秒，完成后会自动重启内核)...")
    get_ipython().run_line_magic("pip", "install -q 'condacolab==0.1.*'")
    import condacolab
    condacolab.install()
    print(f"✅ Condacolab 安装完成 (耗时 {time.perf_counter()-t0:.1f}s)")

In [ ]:
#@title 🚀 2B. 一键部署 GPT-SoVITS（单次 PyTorch + v2全系/v5 底模 + 紧凑进度卡） { display-mode: "form" }
MODEL_PACK = "v2全系 + v5 (测试期推荐: v2/v2Pro/v2ProPlus/v5)" #@param ["v2全系 + v5 (测试期推荐: v2/v2Pro/v2ProPlus/v5)", "仅 v2Pro + v5 (跳过老版 v2)", "全量官方基础包 (含 v1/v2/v3/v4)"]
MODEL_SOURCE = "HF" #@param ["HF", "HF-Mirror", "ModelScope"]
DOWNLOAD_UVR5 = True #@param {type:"boolean"}
FORCE_REINSTALL = False #@param {type:"boolean"}

import os, json, time, html, shutil, subprocess
from collections import deque
from IPython.display import display, HTML

cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
drive_ws = cfg.get("drive_workspace", "")
use_drive = cfg.get("enable_gdrive_cache", False) and os.path.exists("/content/drive/MyDrive")
cache_pro = use_drive and cfg.get("cache_models", False)

TOTAL_STEPS = 5
LOG_FILE = "/content/gsv_setup.log"
CONSTRAINTS_FILE = "/content/gsv_constraints.txt"
timings = []
overall_start = time.perf_counter()
progress_handle = display(HTML("初始化中..."), display_id=True)

def fmt_time(sec):
    m, s = divmod(int(sec), 60)
    return f"{m:02d}:{s:02d} ({sec:.1f}s)"

def update_ui(step_idx, title, status_text, tail_lines=None, color="#38bdf8"):
    pct = int((step_idx / TOTAL_STEPS) * 100)
    total_el = time.perf_counter() - overall_start
    log_box = ""
    if tail_lines:
        safe_lines = "<br>".join(html.escape(x[-110:]) for x in tail_lines)
        log_box = f"<div style='margin-top:8px;padding:6px 10px;background:#020617;border-radius:6px;font-family:monospace;font-size:11px;color:#94a3b8;line-height:1.45;'>{safe_lines}</div>"
    html_str = f"""
    <div style='border:1px solid {color};border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
      <div style='display:flex;justify-content:space-between;margin-bottom:6px;'>
        <b>[{step_idx}/{TOTAL_STEPS}] {title}</b>
        <span>⏱️ 总耗时: {fmt_time(total_el)}</span>
      </div>
      <div style='background:#1e293b;border-radius:6px;height:12px;overflow:hidden;margin-bottom:8px;'>
        <div style='width:{pct}%;background:{color};height:100%;transition:width 0.3s;'></div>
      </div>
      <div style='font-size:13px;color:#cbd5e1;'>{status_text}</div>
      {log_box}
    </div>
    """
    progress_handle.update(HTML(html_str))

def run_cmd(step_idx, title, status_text, cmd, env=None):
    proc = subprocess.Popen(
        ["bash", "-lc", cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    tail = deque(maxlen=4)
    err_tail = deque(maxlen=25)
    last_refresh = 0.0
    with open(LOG_FILE, "a", encoding="utf-8") as lf:
        lf.write(f"\n=== [{step_idx}/{TOTAL_STEPS}] {title} ===\n")
        for line in proc.stdout:
            lf.write(line)
            clean = line.strip()
            if clean and not clean.startswith("\x1b"):
                tail.append(clean)
                err_tail.append(clean)
            now = time.perf_counter()
            if now - last_refresh >= 0.8:
                update_ui(step_idx, title, status_text, list(tail))
                last_refresh = now
    rc = proc.wait()
    if rc != 0:
        update_ui(step_idx, f"{title} (失败)", f"退出码 {rc}，完整日志见 {LOG_FILE}", list(err_tail)[-6:], color="#f87171")
        raise RuntimeError(f"命令失败 (exit={rc})，最近日志:\n" + "\n".join(err_tail))
    return list(tail)

# 写入 pip 版本约束，让 install.sh 一步装对 torch/torchaudio/fastapi，不装两遍
with open(CONSTRAINTS_FILE, "w") as cf:
    cf.write("torch==2.11.0+cu126\ntorchaudio==2.11.0+cu126\nfastapi==0.115.12\nstarlette<0.47.0\n")

env_vars = os.environ.copy()
env_vars["PIP_CONSTRAINT"] = CONSTRAINTS_FILE
env_vars["PIP_EXTRA_INDEX_URL"] = "https://download.pytorch.org/whl/cu126"
if MODEL_SOURCE == "HF-Mirror":
    env_vars["HF_ENDPOINT"] = "https://hf-mirror.com"
if use_drive and cfg.get("hf_cache"):
    env_vars["HF_HOME"] = cfg["hf_cache"]

# Step 1: Clone 源码
t1 = time.perf_counter()
update_ui(1, "获取 GPT-SoVITS 源码", "正在检查 / 克隆 RVC-Boss/GPT-SoVITS...")
if FORCE_REINSTALL and os.path.exists("/content/GPT-SoVITS"):
    shutil.rmtree("/content/GPT-SoVITS")
if not os.path.exists("/content/GPT-SoVITS"):
    run_cmd(1, "获取 GPT-SoVITS 源码", "正在克隆 RVC-Boss/GPT-SoVITS...", "git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS", env=env_vars)
timings.append(("1. 源码准备", time.perf_counter() - t1))

# Step 2: Conda 环境
t2 = time.perf_counter()
update_ui(2, "准备 GPTSoVITS Python 3.10 环境", "检查或创建 /usr/local/envs/GPTSoVITS...")
if not os.path.exists("/usr/local/envs/GPTSoVITS/bin/python"):
    run_cmd(2, "准备 GPTSoVITS Python 3.10 环境", "正在创建 Python 3.10 Conda 环境...", "conda create -n GPTSoVITS python=3.10 -y && source activate GPTSoVITS && pip install -q ipykernel", env=env_vars)
timings.append(("2. Conda 环境", time.perf_counter() - t2))

# Step 3: 运行 install.sh（按选择决定是否跳过 4.25GB 全量底模包）
t3 = time.perf_counter()
marker = "/content/GPT-SoVITS/.colab_installed_ok"
selective_pack = not MODEL_PACK.startswith("全量")
if FORCE_REINSTALL or not os.path.exists(marker):
    sh_path = "/content/GPT-SoVITS/install.sh"
    sh_bak = "/content/GPT-SoVITS/install.sh.orig"
    if not os.path.exists(sh_bak):
        shutil.copy2(sh_path, sh_bak)
    else:
        shutil.copy2(sh_bak, sh_path)
    if selective_pack:
        # 跳过 4.25GB pretrained_models.zip，保留系统依赖、G2PWModel、UVR5 与 Python 包安装
        lines = open(sh_path, encoding="utf-8").splitlines(True)
        new_lines = []
        skip_cont = False
        for ln in lines:
            if skip_cont:
                skip_cont = ln.rstrip().endswith("\\")
                continue
            if "pretrained_models.zip" in ln:
                new_lines.append("    : # skipped full pretrained_models.zip for selective pack\n")
                skip_cont = ln.rstrip().endswith("\\")
                continue
            new_lines.append(ln)
        open(sh_path, "w", encoding="utf-8").write("".join(new_lines))
    uvr_flag = "--download-uvr5" if DOWNLOAD_UVR5 else ""
    desc = "单次安装 PyTorch 2.11+cu126 与依赖（跳过 4.25GB 全量模型包）..." if selective_pack else "单次安装依赖并下载全量预训练模型..."
    run_cmd(3, "安装运行环境与依赖", desc, f"cd /content/GPT-SoVITS && source activate GPTSoVITS && bash install.sh --device CU126 --source {MODEL_SOURCE} {uvr_flag}", env=env_vars)
    open(marker, "w").write("ok")
else:
    update_ui(3, "安装运行环境与依赖", "✅ 基础环境已就绪，跳过重复安装")
timings.append(("3. 依赖与单次 PyTorch", time.perf_counter() - t3))

# Step 4: 按选定组合拉取精简底模（含云盘可选缓存恢复）
t4 = time.perf_counter()
pm_dir = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
os.makedirs(pm_dir, exist_ok=True)
if selective_pack:
    patterns = ["chinese-hubert-base/*", "chinese-roberta-wwm-ext-large/*", "sv/*", "s1v3.ckpt", "v2Pro/*", "gsv-v5-pretrained/*"]
    if "v2全系" in MODEL_PACK:
        patterns.append("gsv-v2final-pretrained/*")
    cache_dir = os.path.join(drive_ws, "models_v2pro_cache") if cache_pro else ""
    if cache_pro and os.path.exists(cache_dir):
        run_cmd(4, "恢复云盘底模缓存", "正在从 Google Drive 增量恢复已缓存的底模...", f"cp -rn {cache_dir}/* {pm_dir}/ 2>/dev/null || true", env=env_vars)
    dl_script = (
        "from huggingface_hub import snapshot_download\n"
        f"snapshot_download(repo_id='lj1995/GPT-SoVITS', local_dir='{pm_dir}', allow_patterns={patterns!r})\n"
    )
    open("/content/dl_models.py", "w").write(dl_script)
    run_cmd(4, f"校验/拉取 {MODEL_PACK}", "正在增量核验并拉取所选底模（跳过 v1 与弃用 v3/v4）...", "/usr/local/envs/GPTSoVITS/bin/python /content/dl_models.py", env=env_vars)
    if cache_pro:
        run_cmd(4, "备份底模到云盘", "正在将当前底模增量同步至 Google Drive...", f"cp -rn {pm_dir}/* {cache_dir}/", env=env_vars)
else:
    update_ui(4, "模型包准备", "✅ 已使用全量官方基础包")
timings.append(("4. 精简底模准备", time.perf_counter() - t4))

# Step 5: 兼容校验 + 云盘工作区联动
t5 = time.perf_counter()
verify_cmd = (
    "/usr/local/envs/GPTSoVITS/bin/python -c \"import torch, torchaudio, fastapi, starlette, gradio; "
    "assert '12.6' in str(torch.version.cuda), torch.version.cuda; "
    "assert fastapi.__version__ == '0.115.12', fastapi.__version__; "
    "print('Verify OK -> torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'torchaudio:', torchaudio.__version__, 'fastapi:', fastapi.__version__, 'starlette:', starlette.__version__)\""
)
try:
    verify_tail = run_cmd(5, "环境校验 & 云盘联动", "正在校验 PyTorch/TorchAudio/FastAPI 版本...", verify_cmd, env=env_vars)
except RuntimeError:
    run_cmd(
        5, "环境校验 & 云盘联动", "检测到版本偏差，执行兜底对齐...",
        "/usr/local/envs/GPTSoVITS/bin/pip install -q torch==2.11.0+cu126 torchaudio==2.11.0+cu126 "
        "--index-url https://download.pytorch.org/whl/cu126 --extra-index-url https://pypi.org/simple && "
        "/usr/local/envs/GPTSoVITS/bin/pip install -q 'fastapi==0.115.12' 'starlette<0.47.0'",
        env=env_vars
    )
    verify_tail = run_cmd(5, "环境校验 & 云盘联动", "再次校验版本...", verify_cmd, env=env_vars)

if use_drive:
    ref_local = "/content/GPT-SoVITS/ref_audios"
    ref_drive = os.path.join(drive_ws, "ref_audios")
    if not os.path.exists(ref_local):
        os.symlink(ref_drive, ref_local)
    wb_dir = os.path.join(drive_ws, "weights_backup")
    if os.path.exists(wb_dir):
        run_cmd(5, "环境校验 & 云盘联动", "正在恢复云盘自训练权重...", f"cp -rn {wb_dir}/* /content/GPT-SoVITS/ 2>/dev/null || true", env=env_vars)
timings.append(("5. 校验与云盘联动", time.perf_counter() - t5))

total_sec = time.perf_counter() - overall_start
rows = "".join(f"<tr><td style='padding:4px 10px;'>{k}</td><td style='padding:4px 10px;text-align:right;'>{fmt_time(v)}</td></tr>" for k, v in timings)
verify_line = html.escape(verify_tail[-1]) if verify_tail else "OK"
progress_handle.update(HTML(f"""
<div style='border:1px solid #22c55e;border-radius:8px;padding:12px 16px;background:#0f172a;color:#e2e8f0;font-family:sans-serif;'>
  <h4 style='margin:0 0 8px 0;color:#22c55e;'>✅ 极速部署完成！(总耗时: {fmt_time(total_sec)})</h4>
  <div style='font-size:12px;color:#94a3b8;margin-bottom:8px;'>{verify_line} | 日志: <code>{LOG_FILE}</code></div>
  <table style='width:100%;max-width:460px;border-collapse:collapse;font-size:13px;'>
    <thead><tr style='border-bottom:1px solid #334155;color:#94a3b8;'><th style='text-align:left;padding:4px 10px;'>阶段</th><th style='text-align:right;padding:4px 10px;'>耗时</th></tr></thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""))

## Step 3. 启动 WebUI（默认启动 TTS 语音合成推理界面）

In [ ]:
#@title 🎙️ 启动选定工具模块（点击日志中的 https://xxxx.gradio.live 即可使用） { display-mode: "form" }
LAUNCH_MODE = "TTS 推理 (inference_webui.py)" #@param ["TTS 推理 (inference_webui.py)", "TTS 并行加速推理 (inference_webui_fast.py)", "完整总控制台 (webui.py)", "UVR5 人声分离与去混响 (tools/uvr5/webui.py)", "数据集文本校对 (tools/subfix_webui.py)"]
MODEL_VERSION = "v2ProPlus" #@param ["v2ProPlus", "v2Pro", "v5turbo", "v5dev", "v2"]
UI_LANGUAGE = "zh_CN" #@param ["zh_CN", "Auto", "en_US"]
ENABLE_PUBLIC_SHARE = True #@param {type:"boolean"}
SUBFIX_LIST_PATH = "output/asr_opt/slicer_opt.list" #@param {type:"string"}

import os, json, time, datetime
cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}

hf_env = f"export HF_HOME='{cfg['hf_cache']}' && " if cfg.get("enable_gdrive_cache") and cfg.get("hf_cache") and os.path.exists("/content/drive/MyDrive") else ""
share_val = "True" if ENABLE_PUBLIC_SHARE else "False"
GPT_MAP = {
    "v2": "GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt",
    "v2Pro": "GPT_SoVITS/pretrained_models/s1v3.ckpt",
    "v2ProPlus": "GPT_SoVITS/pretrained_models/s1v3.ckpt",
    "v5turbo": "GPT_SoVITS/pretrained_models/s1v3.ckpt",
    "v5dev": "GPT_SoVITS/pretrained_models/s1v3.ckpt",
}
SOVITS_MAP = {
    "v2": "GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth",
    "v2Pro": "GPT_SoVITS/pretrained_models/v2Pro/s2Gv2Pro.pth",
    "v2ProPlus": "GPT_SoVITS/pretrained_models/v2Pro/s2Gv2ProPlus.pth",
    "v5turbo": "GPT_SoVITS/pretrained_models/gsv-v5-pretrained/s2Gv5turbo.pth",
    "v5dev": "GPT_SoVITS/pretrained_models/gsv-v5-pretrained/s2Gv5dev.pth",
}
gpt_default = GPT_MAP.get(MODEL_VERSION, "")
sovits_default = SOVITS_MAP.get(MODEL_VERSION, "")
model_env = ""
if gpt_default and sovits_default and os.path.exists(f"/content/GPT-SoVITS/{gpt_default}") and os.path.exists(f"/content/GPT-SoVITS/{sovits_default}"):
    model_env = f"export version={MODEL_VERSION} gpt_path={gpt_default} sovits_path={sovits_default} && "
elif sovits_default:
    print(f"⚠️ 未找到 {gpt_default} 或 {sovits_default}，请确认 Step 2B 的 MODEL_PACK 包含了对应版本；将回退由 WebUI 自动探测权重。")

if "inference_webui_fast.py" in LAUNCH_MODE:
    py_cmd = f"python GPT_SoVITS/inference_webui_fast.py {UI_LANGUAGE}"
elif "inference_webui.py" in LAUNCH_MODE:
    py_cmd = f"python GPT_SoVITS/inference_webui.py {UI_LANGUAGE}"
elif "uvr5/webui.py" in LAUNCH_MODE:
    py_cmd = f"python tools/uvr5/webui.py \"cuda\" True 9873 {share_val}"
elif "subfix_webui.py" in LAUNCH_MODE:
    py_cmd = f"python tools/subfix_webui.py --load_list \"{SUBFIX_LIST_PATH}\" --webui_port 9871 --is_share {share_val}"
else:
    py_cmd = f"python webui.py {UI_LANGUAGE}"

print(f"🕒 启动时间: {datetime.datetime.now().strftime('%H:%M:%S')} | 模式: {LAUNCH_MODE} | 默认底模: {MODEL_VERSION}")
print("💡 进入 TTS WebUI 后，也可直接在页面左上角切换 v2 / v2Pro / v2ProPlus / v5turbo / v5dev（注意 GPT 与 SoVITS 版本对应）。")
if cfg.get("enable_gdrive_cache"):
    print(f"💾 云盘参考音频目录已映射到: /content/GPT-SoVITS/ref_audios -> {cfg['drive_workspace']}/ref_audios")
print("⏳ 等待下方打印出 https://xxxx.gradio.live 公网链接后点击打开（不用时记得点停止释放 GPU 额度）...\n")

t_run = time.perf_counter()
try:
    get_ipython().system(f"cd /content/GPT-SoVITS && source activate GPTSoVITS && {hf_env}{model_env}export PYTHONPATH=/content/GPT-SoVITS:${{PYTHONPATH:-}} && export is_share={share_val} && {py_cmd}")
finally:
    dt = time.perf_counter() - t_run
    print(f"\n🛑 WebUI 已停止，本次运行时长: {int(dt//60):02d}分{int(dt%60):02d}秒")

## Step 4. 备份成果到云盘 & 在线试听生成音频
> 停掉 Step 3 后运行下方单元格，可一键把生成的语音和训练权重备份到 Google Drive，并直接在网页里试听。

In [ ]:
#@title 💾 同步输出/权重到云盘 并试听最新生成的音频 { display-mode: "form" }
MAX_PREVIEW_FILES = 5 #@param {type:"integer"}

import os, glob, json, time, shutil
from IPython.display import display, HTML, Audio

t0 = time.perf_counter()
cfg_path = "/content/.gsv_colab_config.json"
cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
gsv_root = "/content/GPT-SoVITS"

if cfg.get("enable_gdrive_cache") and os.path.exists("/content/drive/MyDrive"):
    drive_ws = cfg["drive_workspace"]
    out_local = os.path.join(gsv_root, "output")
    out_drive = os.path.join(drive_ws, "outputs")
    copied_out = 0
    if os.path.exists(out_local):
        for root, _, files in os.walk(out_local):
            for fn in files:
                src = os.path.join(root, fn)
                rel = os.path.relpath(src, out_local)
                dst = os.path.join(out_drive, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
                    shutil.copy2(src, dst)
                    copied_out += 1
    copied_w = 0
    wb_dir = os.path.join(drive_ws, "weights_backup")
    for folder in glob.glob(os.path.join(gsv_root, "*_weights*")):
        for root, _, files in os.walk(folder):
            for fn in files:
                if fn.endswith((".pth", ".ckpt")):
                    src = os.path.join(root, fn)
                    rel = os.path.relpath(src, gsv_root)
                    dst = os.path.join(wb_dir, rel)
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
                        shutil.copy2(src, dst)
                        copied_w += 1
    print(f"☁️ 云盘同步完成 (耗时 {time.perf_counter()-t0:.2f}s): 新增输出音频 {copied_out} 个，模型权重 {copied_w} 个 -> {drive_ws}")
else:
    print("ℹ️ 未挂载 Google Drive，跳过云盘同步。")

audio_files = []
for ext in ("*.wav", "*.flac", "*.mp3"):
    audio_files.extend(glob.glob(os.path.join(gsv_root, "output", "**", ext), recursive=True))
audio_files.sort(key=os.path.getmtime, reverse=True)

if not audio_files:
    print("📭 output 目录暂无生成的音频文件。")
else:
    print(f"🎧 找到 {len(audio_files)} 个音频，展示最近 {min(MAX_PREVIEW_FILES, len(audio_files))} 个：")
    for p in audio_files[:MAX_PREVIEW_FILES]:
        sz = os.path.getsize(p) / 1024
        display(HTML(f"<div style='margin-top:8px;font-size:13px;'><b>{os.path.basename(p)}</b> ({sz:.1f} KB) — <code>{p}</code></div>"))
        display(Audio(p))

## 📚 内置工具模块速查

| 启动入口 / 标签页 | 核心用途 | 使用提示 |
| :--- | :--- | :--- |
| **`GPT_SoVITS/inference_webui.py`** | **TTS 语音合成与克隆推理（标准版）** | 传 3~10 秒干声作参考音频即可配音；推荐搭配 `v2` 或 `v2Pro` / `v2ProPlus`；不确定采样参数时 `top_k / top_p / temperature` 保持默认。 |
| **`GPT_SoVITS/inference_webui_fast.py`** | **TTS 并行加速推理** | 支持批量与并行推理加速，适合一次合成较长文本或多句配音。 |
| **`tools/uvr5/webui.py`** | **UVR5 人声分离 & 去混响/去回声** | 提取干声轨可清除伴奏与混响；若提取非干声/混响残差轨（如 `MDX-Net` / `DeEcho`），会保留空间尾音与延迟回响，听感上类似音尾被拉长。 |
| **`tools/subfix_webui.py`** | **数据集文本校对与打标** | 配合 `--load_list`（默认 `output/asr_opt/slicer_opt.list`）逐条试听切片并修正 ASR 文本。 |
| **`webui.py` -> `0b-语音切分`** | **长音频自动切段** | 按静音把长录音批量切成 4~10 秒片段，用于挑选参考音频或制作训练集。 |
| **`webui.py` -> `0c-降噪` / `0d-ASR`** | **语音降噪 & 自动识别打标** | 批量降噪并调用 ASR 生成带文本标注的 `.list` 文件，供 `1A/1B` 微调训练使用。 |
| **`webui.py` -> `1A/1B-微调训练`** | **专属音色模型训练** | 1~5 分钟干净音频即可微调专属 GPT/SoVITS 权重（`v2` / `v2Pro` 均可稳跑 T4）。 |
| **`webui.py` -> `2-Voice Changer`** | **音色转换（变声器）** | 保留原音频语气与节奏，将人声替换为目标音色。 |